# Library

In [7]:
import sys
sys.path.append("/project/assistant")

%load_ext autoreload
%autoreload 2

import os

import pandas as pd

from db import ConversationLog

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Database Logging

Elasticsearch stores the knowledge base; PostgreSQL stores the system history. The database follows a small star-shaped schema: `conversations` is the central fact table, with one row per answered question, while `feedback` and `judgements` reference it through `conversation_id`.

```text
conversations   <- one row per answered question:
    ^    ^         question, answer, model, tokens, cost,
    |    |         retrieval time, total time, sources (JSONB),
    |    |         session, channel, timestamp
    |    |
feedback  judgements
(human,   (machine: the LLM judge's verdict and reasoning
 sparse)   for each production answer)
```

Every question answered by the application creates one `conversations` row. The evaluator then writes one `judgements` row automatically, giving machine evaluation complete coverage. `feedback` is optional and appears only when the user submits a thumbs-up or thumbs-down, so human evaluation remains sparse. Grafana joins the three tables through `conversation_id` to build the monitoring dashboards.

The module follows the same package pattern used elsewhere: `ConversationLog`, imported from `assistant/db.py`, receives its database connection settings at construction time.

`create_tables()` is idempotent because `make init-db` runs it on every fresh setup. Schema changes use the explicit destructive path, `recreate=True`, exposed through `make reset-db`, mirroring the reset behavior of the Elasticsearch indexer.

A `source` column records which interface produced each conversation, allowing future entry points,such as a Telegram bot or editor plugin,to reuse the same persistence and observability layer.


In [8]:
log = ConversationLog(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    user=os.getenv("POSTGRES_USER", "user"),
    password=os.getenv("POSTGRES_PASSWORD", "pswd"),
    dbname=os.getenv("APP_POSTGRES_DB", "obsidian_assistant"),
)

with log._connect() as conn:
    version = conn.execute("SELECT version()").fetchone()[0]
print(version[:60])

PostgreSQL 18.4 (Debian 18.4-1.pgdg13+1) on x86_64-pc-linux-


In [9]:
log.create_tables()
log.create_tables()   # twice on purpose: idempotence is the contract

with log._connect() as conn:
    tables = conn.execute(
        "SELECT tablename FROM pg_tables WHERE schemaname = 'public'"
    ).fetchall()
print("tables:", [t[0] for t in tables])

tables: ['conversations', 'feedback', 'judgements']


The full schema, table by table: every column with its type and
whether it accepts nulls. The nullable columns are the ones whose
feeder is optional (a comment nobody wrote, a session that a future
channel may not have).

In [10]:
with log._connect() as conn:
    schema = pd.DataFrame(conn.execute(
        """SELECT table_name, column_name, data_type, is_nullable
           FROM information_schema.columns
           WHERE table_schema = 'public'
           ORDER BY table_name, ordinal_position"""
    ).fetchall(), columns=["table", "column", "type", "nullable"])

for table in schema["table"].unique():
    print(f"== {table} ==")
    part = schema[schema.table == table]
    for _, r in part.iterrows():
        print(f"  {r['column']:<18} {r['type']:<26} null: {r['nullable']}")
    print()

== conversations ==
  id                 integer                    null: NO
  question           text                       null: NO
  answer             text                       null: NO
  model              text                       null: NO
  embed_model        text                       null: YES
  search_mode        text                       null: NO
  num_sources        integer                    null: YES
  sources            jsonb                      null: YES
  prompt_tokens      integer                    null: YES
  completion_tokens  integer                    null: YES
  total_tokens       integer                    null: YES
  cost               numeric                    null: YES
  retrieval_time     real                       null: YES
  response_time      real                       null: YES
  source             text                       null: NO
  session_id         text                       null: YES
  created_at         timestamp with time zone   null: NO



A fake conversation exercises the whole write path: the conversation
row with every enriched field, the judge verdict and the human thumb
pointing at it. Everything is deleted at the end, so the diary keeps
only real usage.

In [11]:
fake_id = log.save_conversation(
    question="fake question, delete me",
    answer="fake answer",
    model="test-model",
    prompt_tokens=100,
    completion_tokens=20,
    cost=0.000123,
    response_time=1.5,
    embed_model="test-embeddings",
    search_mode="hybrid",
    num_sources=2,
    sources=[{"path": "a.md", "start": 0, "score": 0.9},
             {"path": "b.md", "start": 1000, "score": 0.5}],
    retrieval_time=0.12,
    session_id="fake-session",
)
log.save_feedback(fake_id, 1, comment="fake comment")
log.save_judgement(fake_id, "RELEVANT", "fake reasoning, delete me",
                   judge_model="test-judge")
print("conversation id:", fake_id)

conversation id: 2


In [12]:
with log._connect() as conn:
    convs = conn.execute(
        """SELECT id, question, model, embed_model, search_mode, num_sources,
                  total_tokens, cost, retrieval_time, response_time, session_id
           FROM conversations ORDER BY id DESC LIMIT 3"""
    ).fetchall()
    fb = conn.execute(
        "SELECT id, conversation_id, thumbs, comment FROM feedback ORDER BY id DESC LIMIT 3"
    ).fetchall()
    jd = conn.execute(
        "SELECT id, conversation_id, relevance, judge_model FROM judgements ORDER BY id DESC LIMIT 3"
    ).fetchall()

print("conversations:")
for row in convs:
    print("  ", row)
print("feedback:")
for row in fb:
    print("  ", row)
print("judgements:")
for row in jd:
    print("  ", row)

conversations:
   (2, 'fake question, delete me', 'test-model', 'test-embeddings', 'hybrid', 2, 120, Decimal('0.000123'), 0.12, 1.5, 'fake-session')
   (1, 'What is the agentic loop?', 'gpt-5.4-mini', 'all-MiniLM-L6-v2', 'hybrid', 10, 4899, Decimal('0.001356'), 0.143018, 3.3244855, '5ac46a1b-5d23-4e60-bd0c-71a98c1262cb')
feedback:
   (2, 2, 1, 'fake comment')
   (1, 1, 1, None)
judgements:
   (2, 2, 'RELEVANT', 'test-judge')
   (1, 1, 'RELEVANT', 'gpt-5.4-mini')


How the star connects: one join by conversation_id produces the
unified view, one line per answered question with the llm's numbers,
the judge's verdict and the human vote side by side. This is exactly
the query shape the grafana dashboards are built on. LEFT JOIN
matters: a conversation without feedback (most of them) must still
appear.

In [13]:
with log._connect() as conn:
    rows = conn.execute(
        """SELECT c.id, left(c.question, 40) AS question, c.model,
                  c.cost, c.response_time, j.relevance, f.thumbs
           FROM conversations c
           LEFT JOIN judgements j ON j.conversation_id = c.id
           LEFT JOIN feedback   f ON f.conversation_id = c.id
           ORDER BY c.id DESC LIMIT 5"""
    ).fetchall()

pd.DataFrame(rows, columns=["id", "question", "model", "cost",
                            "response_time", "judge", "thumbs"])

,id,question,model,cost,response_time,judge,thumbs
0,2,"fake question, delete me",test-model,0.000123,1.500000,RELEVANT,1
1,1,What is the agentic loop?,gpt-5.4-mini,0.001356,3.324486,RELEVANT,1


In [14]:
# remove the fake rows (children first, they reference the
# conversation); real app conversations are left untouched
with log._connect() as conn:
    conn.execute("DELETE FROM judgements WHERE conversation_id = %s", (fake_id,))
    conn.execute("DELETE FROM feedback WHERE conversation_id = %s", (fake_id,))
    conn.execute("DELETE FROM conversations WHERE id = %s", (fake_id,))
    counts = conn.execute(
        """SELECT (SELECT count(*) FROM conversations),
                  (SELECT count(*) FROM feedback),
                  (SELECT count(*) FROM judgements)"""
    ).fetchone()
print(f"conversations: {counts[0]} | feedback: {counts[1]} | judgements: {counts[2]}")

conversations: 1 | feedback: 1 | judgements: 1
